# Paper 3 — Train β=1 and β=4 on dSprites
Run cells sequentially on Colab GPU (A100 or T4). No automatic dataset downloads. The two **long** training cells are separate and never run from setup. See README for scientific assumptions.

In [ ]:
# Clone private source without embedding or printing the access token.
from pathlib import Path
import os, subprocess, tempfile, stat
from google.colab import userdata

REPO = Path('/content/advanced-ml-paper-reproductions')
PROJECT = REPO / 'paper3-beta-vae'
if not PROJECT.exists():
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets and grant notebook access.')
    with tempfile.TemporaryDirectory() as tmp:
        askpass = Path(tmp) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "x-access-token" ;;\n  *Password*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
        subprocess.run(['git','clone','--quiet', 'https://github.com/vepaan/advanced-ml-paper-reproductions.git',str(REPO)], env=env, check=True)
    del token
    print('Cloned private repository to', REPO)
else:
    print('Reusing existing project checkout:', PROJECT)
assert (PROJECT / 'src' / 'models.py').is_file(), 'Push paper3-beta-vae/ into GitHub first and rerun setup.'
%cd /content/advanced-ml-paper-reproductions/paper3-beta-vae
# Colab usually ships matplotlib. Install if missing; do NOT replace Colab torch with old Windows pins.
import importlib.util
if importlib.util.find_spec('matplotlib') is None:
    subprocess.run(['python','-m','pip','install','-q','matplotlib'],check=True)
import torch
print('torch:',torch.__version__,'CUDA:',torch.cuda.is_available(),
      'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
FNAME = 'dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'
DATA_SRC = Path('/content/drive/MyDrive/ML-Paper-Reproductions/datasets/dsprites') / FNAME
DATA_DIR = Path('/content/datasets/dsprites')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA = DATA_DIR / FNAME
if not DATA.exists():
    if not DATA_SRC.is_file():
        raise FileNotFoundError(f'Upload the official downloaded NPZ into Google Drive first: {DATA_SRC}')
    shutil.copy2(DATA_SRC, DATA)
    print('Copied local official dataset from Drive.')
else:
    print('Using existing local dataset file.')
assert DATA.stat().st_size > 1_000_000, 'Unexpectedly small NPZ. Check your manual download.'
LOCAL_RUNS=Path('/content/paper3_runs')
DRIVE_RUNS=Path('/content/drive/MyDrive/ML-Paper-Reproductions/paper3')
LOCAL_RUNS.mkdir(parents=True,exist_ok=True)
DRIVE_RUNS.mkdir(parents=True,exist_ok=True)
print('NPZ:', DATA, f'{DATA.stat().st_size/1024**2:.1f} MiB')


In [ ]:
import subprocess, sys
assert torch.cuda.is_available(), 'Select a GPU runtime before full training.'
# First extraction makes a 3 GB disk-backed .npy file. This is ONCE per runtime.
cmd=[sys.executable,'-m','src.validate','--paper-model','--data',str(DATA),
     '--benchmark','--device','cuda','--batch-size','256','--max-steps','12000',
     '--warmup','3','--measured','12']
subprocess.run(cmd,check=True)
print('Review the printed measured time estimate before running the next cell.')


## Standard VAE (Figure 7B): β = 1
One seed (2026); initial weights and shuffled minibatches match β=4. Adagrad 0.01; look at real benchmark before running. This cell resumes from a saved checkpoint when rerun.

In [ ]:
import subprocess, sys
MAX_STEPS = 12000   # Total target updates, not extra updates on resume; increase if evidence warrants.
BATCH_SIZE = 256
cmd=[sys.executable,'-m','src.train','--beta','1','--data',str(DATA),
     '--output-root',str(LOCAL_RUNS),'--backup-root',str(DRIVE_RUNS),
     '--max-steps',str(MAX_STEPS),'--batch-size',str(BATCH_SIZE),
     '--learning-rate','0.01','--seed','2026','--ckpt-every','2000','--log-every','200']
subprocess.run(cmd,check=True)


## β-VAE (Figure 7A): β = 4
Identical initialization and batch order; only KL multiplier changes. One seed is insufficient to establish robustness.

In [ ]:
import subprocess, sys
cmd=[sys.executable,'-m','src.train','--beta','4','--data',str(DATA),
     '--output-root',str(LOCAL_RUNS),'--backup-root',str(DRIVE_RUNS),
     '--max-steps',str(MAX_STEPS),'--batch-size',str(BATCH_SIZE),
     '--learning-rate','0.01','--seed','2026','--ckpt-every','2000','--log-every','200']
subprocess.run(cmd,check=True)


In [ ]:
import torch
for beta in (1,4):
    local = LOCAL_RUNS / f'beta{beta}' / 'model.pt'
    assert local.exists(), f'Missing {local}'
    c=torch.load(local,map_location='cpu',weights_only=False)
    assert c['beta']==beta and c['step']>=MAX_STEPS
    print(f'β={beta} step={c["step"]} seed={c["seed"]} inference state={local}')
print('Next: run reproduce_figure7.ipynb (no training required).')
